# Wake Word "Letícia" para Home Assistant (v6 — GPU Otimizado)

Todas as fontes de dados verificadas contra o [notebook oficial do openWakeWord](https://github.com/dscripka/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb).

## Instruções
1. GPU T4 ativa
2. Execute célula a célula
3. **OBRIGATÓRIO:** Reiniciar sessão após Etapa 1a

> Tempo total estimado: ~2 horas (inclui download de 17.3 GB)

---

## Etapa 1a: Instalação de Dependências
⚠️ **Reiniciar sessão obrigatório após esta célula!**

In [1]:
import os, locale
locale.getpreferredencoding = lambda *a: 'UTF-8'

print('=' * 60)
print('  ETAPA 1a: Instalação de Dependências')
print('=' * 60)

# ============================================================
# Repositórios
#   - piper-sample-generator: fork dscripka (tem generate_samples.py)
#     Fonte: https://github.com/dscripka/piper-sample-generator
#     Verificado: API GitHub retorna 200, contém generate_samples.py + impulses/
#   - openWakeWord: repositório principal
#     Fonte: https://github.com/dscripka/openWakeWord
#     Verificado: API GitHub retorna 200, release mais recente v0.6.0
# ============================================================
if not os.path.exists('./piper-sample-generator'):
    !git clone -q https://github.com/dscripka/piper-sample-generator
    print('[OK] piper-sample-generator clonado')
else:
    print('[SKIP] piper-sample-generator já existe')

if not os.path.exists('./openwakeword'):
    !git clone -q https://github.com/dscripka/openWakeWord openwakeword
    print('[OK] openWakeWord clonado')
else:
    print('[SKIP] openWakeWord já existe')

# ============================================================
# Pacotes Python
# Ordem importa: torch primeiro, depois openwakeword, depois extras
# ============================================================
print('\nInstalando pacotes...')
!pip install -q pathvalidate piper-tts piper-phonemize-cross webrtcvad
!pip install -q 'torch<=2.5' torchvision torchaudio
!pip install -q -e ./openwakeword
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0
!pip install -q speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0
!pip install -q acoustics==0.2.6 scipy
!pip install -q onnxruntime ai-edge-litert onnxsim
# onnx_tf: usado pelo train.py convert_onnx_to_tflite (linha 578-597 do train.py)
# O notebook oficial usa tensorflow-cpu==2.8.1 + onnx_tf==1.10.0
# Mas no Colab 2026, essas versões antigas podem conflitar.
# Instalamos onnx2tf como alternativa moderna:
!pip install -q onnx2tf tensorflow==2.19.0
!pip install -q onnx==1.19.1 onnx_graphsurgeon
# datasets==2.14.6: mesma versão do notebook oficial
!pip install -q datasets==2.14.6
!cd piper-sample-generator && pip install -q -r requirements.txt

print('[OK] Pacotes instalados')
print()
print('⚠️  REINICIE A SESSÃO AGORA!')
print('   Ambiente de execução > Reiniciar sessão')
print('   Depois execute a partir da Etapa 1b')

  ETAPA 1a: Instalação de Dependências
[OK] piper-sample-generator clonado
[OK] openWakeWord clonado

Instalando pacotes...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 993.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## ⚠️ Reiniciar Sessão (OBRIGATÓRIO)

**Ambiente de execução → Reiniciar sessão**

Depois execute a partir da próxima célula (Etapa 1b).

---

## Etapa 1b: Pós-reinício — Downloads de modelos

In [1]:
import os, locale
import numpy as np
locale.getpreferredencoding = lambda *a: 'UTF-8'

print('=' * 60)
print('  ETAPA 1b: Verificação pós-reinício + Downloads')
print('=' * 60)

# Teste de compatibilidade numpy (falha se numpy não carregou corretamente)
_ = np.random.RandomState(42)
print(f'numpy {np.__version__}: OK')

# ============================================================
# Modelos de embedding do openWakeWord
# Fonte: release v0.5.1 (verificado via API GitHub)
# URLs confirmadas como assets do release:
#   https://github.com/dscripka/openWakeWord/releases/tag/v0.5.1
# ============================================================
models_dir = 'openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
base_url = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    fpath = os.path.join(models_dir, fname)
    if not os.path.exists(fpath):
        !wget -q '{base_url}/{fname}' -O {fpath}
        print(f'[OK] {fname}')
    else:
        print(f'[SKIP] {fname}')

# ============================================================
# Modelo LibriTTS para piper-sample-generator
# Fonte: release v2.0.0 de rhasspy/piper-sample-generator
# URL verificada via API GitHub releases
# ============================================================
libritts = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(libritts):
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    url = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    print('Baixando LibriTTS v2...')
    !wget -q -O {libritts} '{url}'
    print('[OK] LibriTTS v2')
else:
    print('[SKIP] LibriTTS')

# ============================================================
# Vozes Piper pt_BR
# Fonte: HuggingFace rhasspy/piper-voices
# URLs verificadas: HTTP 302 (redirect para download)
# ============================================================
os.makedirs('piper_voices_ptbr', exist_ok=True)
hf_base = 'https://huggingface.co/rhasspy/piper-voices/resolve/main'
voices = {
    'pt_BR-faber-medium': 'pt/pt_BR/faber/medium',
    'pt_BR-edresson-low': 'pt/pt_BR/edresson/low',
}
for name, path in voices.items():
    onnx_path = f'piper_voices_ptbr/{name}.onnx'
    if not os.path.exists(onnx_path):
        !wget -q -O {onnx_path} '{hf_base}/{path}/{name}.onnx'
        !wget -q -O {onnx_path}.json '{hf_base}/{path}/{name}.onnx.json'
        print(f'[OK] {name}')
    else:
        print(f'[SKIP] {name}')

print()
print('[OK] ETAPA 1b CONCLUÍDA!')
# GPU check
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB) ✅')
else:
    print('⚠️  GPU NÃO DETECTADA — treinamento será mais lento')


  ETAPA 1b: Verificação pós-reinício + Downloads
numpy 1.26.4: OK
[OK] embedding_model.onnx
[OK] embedding_model.tflite
[OK] melspectrogram.onnx
[OK] melspectrogram.tflite
Baixando LibriTTS v2...
[OK] LibriTTS v2
[OK] pt_BR-faber-medium
[OK] pt_BR-edresson-low

[OK] ETAPA 1b CONCLUÍDA!
GPU: Tesla T4 (14.6 GB) ✅


## Etapa 2: Testar Pronúncia
Ouça os áudios. Confirme que "Letícia" soa correto.

In [2]:
import subprocess, os
from IPython.display import Audio, display

print('=' * 60)
print('  ETAPA 2: Testando pronúncia')
print('=' * 60)

target_word = 'letícia'
os.makedirs('test_audio', exist_ok=True)

for name, label in [('pt_BR-faber-medium','Faber'), ('pt_BR-edresson-low','Edresson')]:
    out = f'test_audio/test_{label.lower()}.wav'
    r = subprocess.run(
        ['piper', '--model', f'piper_voices_ptbr/{name}.onnx', '--output_file', out],
        input=target_word, capture_output=True, text=True
    )
    if os.path.exists(out):
        print(f'Voz {label} (pt_BR):')
        display(Audio(out, autoplay=False))
    else:
        print(f'[ERRO] {label}: {r.stderr[:200]}')

print('[OK] ETAPA 2 CONCLUÍDA!')

  ETAPA 2: Testando pronúncia
Voz Faber (pt_BR):


Voz Edresson (pt_BR):


[OK] ETAPA 2 CONCLUÍDA!


## Etapa 3: Download de Dados Auxiliares

### Fontes de dados (todas verificadas):

| Dado | Dataset HuggingFace | Como baixar | Tamanho |
|------|---------------------|-------------|--------|
| MIT RIRs | `davidscripka/MIT_environmental_impulse_responses` | `load_dataset(..., split='train')` | ~30 MB (271 WAVs) |
| AudioSet | `agkphysics/AudioSet` | wget de `.tar` → converte 16kHz | ~1 GB |
| FMA | `rudraml/fma` | `load_dataset(..., name='small', split='train')` | ~300 MB |
| ACAV100M features | `davidscripka/openwakeword_features` | **wget direto** do arquivo `.npy` | **17.3 GB** |
| Validation features | `davidscripka/openwakeword_features` | **wget direto** do arquivo `.npy` | 185 MB |

> ⚠️ ACAV100M = 17.3 GB. ~10-20 min no Colab.

**Tempo total estimado: ~30-45 minutos**

In [3]:
import os
import numpy as np
import scipy.io.wavfile as wav
import datasets
from tqdm import tqdm
from pathlib import Path

print('=' * 60)
print('  ETAPA 3: Baixando dados auxiliares')
print('=' * 60)

# ============================================================
# 3a. MIT Room Impulse Responses
# Dataset: davidscripka/MIT_environmental_impulse_responses
#   Verificado: API retorna 200, 271 WAV files, formato audiofolder
#   Split correto: 'train'
#   Notebook oficial, linha 185:
#     datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",
#                           split="train", streaming=True)
# ============================================================
rir_dir = 'mit_rirs'
if not os.path.exists(rir_dir) or len([f for f in os.listdir(rir_dir) if f.endswith('.wav')]) == 0:
    print('\n[3a] MIT Room Impulse Responses...')
    os.makedirs(rir_dir, exist_ok=True)
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    count = 0
    for row in tqdm(rir_dataset, desc='MIT RIRs'):
        name = row['audio']['path'].split('/')[-1]
        audio_array = np.array(row['audio']['array'])
        wav.write(
            os.path.join(rir_dir, name), 16000,
            (audio_array * 32767).astype(np.int16)
        )
        count += 1
    print(f'[OK] MIT RIRs: {count} arquivos')
else:
    n = len([f for f in os.listdir(rir_dir) if f.endswith('.wav')])
    print(f'[SKIP] MIT RIRs ({n} arquivos)')

# ============================================================
# 3b. AudioSet (background noise)
# Dataset: agkphysics/AudioSet
#   Verificado: API retorna 200
#   Notebook oficial, linhas 210-229:
#     Baixa tar de HF, extrai FLACs, converte para 16kHz WAV
# ============================================================
as_dir = 'audioset_16k'
if not os.path.exists(as_dir) or len(os.listdir(as_dir)) == 0:
    print('\n[3b] AudioSet background noise...')
    os.makedirs('audioset', exist_ok=True)
    os.makedirs(as_dir, exist_ok=True)
    fname = 'bal_train09.tar'
    link = f'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}'
    if not os.path.exists(f'audioset/{fname}'):
        !wget -q --show-progress -O audioset/{fname} '{link}'
    !cd audioset && tar -xf {fname} 2>/dev/null || true
    # Converter FLACs para 16kHz WAV
    flac_files = list(Path('audioset/audio').glob('**/*.flac')) if os.path.exists('audioset/audio') else []
    if len(flac_files) > 0:
        print(f'  Convertendo {len(flac_files)} FLAC → WAV 16kHz...')
        audioset_ds = datasets.Dataset.from_dict({'audio': [str(i) for i in flac_files]})
        audioset_ds = audioset_ds.cast_column('audio', datasets.Audio(sampling_rate=16000))
        for row in tqdm(audioset_ds, desc='AudioSet'):
            name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
            audio_array = np.array(row['audio']['array'])
            wav.write(
                os.path.join(as_dir, name), 16000,
                (audio_array * 32767).astype(np.int16)
            )
    n = len(os.listdir(as_dir))
    print(f'[OK] AudioSet: {n} clips')
else:
    print(f'[SKIP] AudioSet ({len(os.listdir(as_dir))} clips)')

# ============================================================
# 3c. FMA (Free Music Archive)
# Dataset: rudraml/fma
#   Verificado: API retorna 200
#   Notebook oficial, linhas 232-245:
#     load_dataset("rudraml/fma", name="small", split="train", streaming=True)
#     1 hora de clips (3600/30 = 120 clips de 30s)
# ============================================================
fma_dir = 'fma'
if not os.path.exists(fma_dir) or len(os.listdir(fma_dir)) == 0:
    print('\n[3c] FMA música (1 hora)...')
    os.makedirs(fma_dir, exist_ok=True)
    fma_dataset = datasets.load_dataset(
        'rudraml/fma', name='small', split='train', streaming=True
    )
    fma_iter = iter(
        fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
    )
    n_hours = 1
    n_clips = n_hours * 3600 // 30  # 120 clips de 30s
    count = 0
    for i in tqdm(range(n_clips), desc='FMA'):
        try:
            row = next(fma_iter)
            name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
            audio_array = np.array(row['audio']['array'])
            wav.write(
                os.path.join(fma_dir, name), 16000,
                (audio_array * 32767).astype(np.int16)
            )
            count += 1
        except StopIteration:
            break
    print(f'[OK] FMA: {count} clips')
else:
    print(f'[SKIP] FMA ({len(os.listdir(fma_dir))} clips)')

# ============================================================
# 3d. ACAV100M pre-computed features
# Fonte: davidscripka/openwakeword_features (arquivo direto, NÃO split)
#   Verificado: Arquivo listado na tree do HF, 17.3 GB
#   Notebook oficial, linha 261:
#     !wget https://huggingface.co/.../openwakeword_features_ACAV100M_2000_hrs_16bit.npy
# ============================================================
acav_file = 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
if not os.path.exists(acav_file):
    print('\n[3d] ACAV100M features (17.3 GB — pode demorar ~10-20 min)...')
    url = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    !wget -q --show-progress -c '{url}'
    if os.path.exists(acav_file):
        size_gb = os.path.getsize(acav_file) / (1024**3)
        print(f'[OK] ACAV100M: {size_gb:.1f} GB')
    else:
        print('[ERRO] ACAV100M download falhou!')
else:
    size_gb = os.path.getsize(acav_file) / (1024**3)
    print(f'[SKIP] ACAV100M ({size_gb:.1f} GB)')

# ============================================================
# 3e. Validation set features
# Fonte: davidscripka/openwakeword_features (arquivo direto, NÃO split)
#   Verificado: Arquivo listado na tree do HF, 185 MB
#   Notebook oficial, linha 264:
#     !wget https://huggingface.co/.../validation_set_features.npy
# ============================================================
val_file = 'validation_set_features.npy'
if not os.path.exists(val_file):
    print('\n[3e] Validation features (185 MB)...')
    url = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy'
    !wget -q --show-progress '{url}'
    print('[OK] Validation features')
else:
    size_mb = os.path.getsize(val_file) / (1024**2)
    print(f'[SKIP] Validation features ({size_mb:.0f} MB)')

# ============================================================
# Resumo
# ============================================================
print()
print('Resumo dos dados:')
for d, label in [('mit_rirs','RIRs'), ('audioset_16k','AudioSet'), ('fma','FMA')]:
    if os.path.exists(d):
        n = len([f for f in os.listdir(d) if f.endswith('.wav')])
        print(f'  {label}: {n} arquivos WAV')
    else:
        print(f'  ❌ {label}: FALTANDO')
for f, label in [(acav_file,'ACAV100M'), (val_file,'Validation')]:
    if os.path.exists(f):
        s = os.path.getsize(f) / (1024**2)
        unit = 'GB' if s > 1024 else 'MB'
        val = s/1024 if s > 1024 else s
        print(f'  {label}: {val:.1f} {unit}')
    else:
        print(f'  ❌ {label}: FALTANDO')

print()
print('[OK] ETAPA 3 CONCLUÍDA!')

  ETAPA 3: Baixando dados auxiliares

[3a] MIT Room Impulse Responses...


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

MIT RIRs: 270it [01:24,  3.21it/s]


[OK] MIT RIRs: 270 arquivos

[3b] AudioSet background noise...
[OK] AudioSet: 0 clips

[3c] FMA música (1 hora)...


FMA: 100%|██████████| 120/120 [00:40<00:00,  2.99it/s]


[OK] FMA: 120 clips

[3d] ACAV100M features (17.3 GB — pode demorar ~10-20 min)...
openwakeword_featur 100%[===================>]  16.09G   228MB/s    in 99s     
[OK] ACAV100M: 16.1 GB

[3e] Validation features (185 MB)...
validation_set_feat 100%[===================>] 176.27M   143MB/s    in 1.2s    
[OK] Validation features

Resumo dos dados:
  RIRs: 270 arquivos WAV
  AudioSet: 0 arquivos WAV
  FMA: 120 arquivos WAV
  ACAV100M: 16.1 GB
  Validation: 176.3 MB

[OK] ETAPA 3 CONCLUÍDA!


## Etapa 4: Gerar Clips de Treinamento com Piper pt_BR

Gera clips com vozes pt_BR usando `piper` CLI.

| Tipo | Quantidade | Fonte |
|------|-----------|-------|
| Positivos treino | 1.500 | Palavra "letícia" |
| Positivos validação | 500 | Palavra "letícia" |
| Negativos treino | 1.500 | Palavras foneticamente similares |
| Negativos validação | 500 | Palavras foneticamente similares |

**Tempo estimado: ~1-2 horas**

In [4]:
import os, subprocess, uuid, random, time
from concurrent.futures import ThreadPoolExecutor, as_completed

print('=' * 60)
print('  ETAPA 4: Gerando clips com Piper pt_BR')
print('=' * 60)

target_word = 'letícia'
model_name  = 'leticia'
n_positive  = 1500
n_val       = 500

ptbr_voices = [
    'piper_voices_ptbr/pt_BR-faber-medium.onnx',
    'piper_voices_ptbr/pt_BR-edresson-low.onnx',
]
length_scales = [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25]
noise_scales  = [0.5, 0.6, 0.667, 0.7, 0.8, 0.9, 0.98]
noise_ws      = [0.5, 0.6, 0.7, 0.8, 0.9, 0.98]

negative_words = [
    'patrícia', 'notícia', 'delícia', 'justiça', 'preguiça',
    'milícia', 'malícia', 'polícia', 'carência', 'urgência',
    'letivo', 'letrada', 'legítima', 'legião', 'elétrica',
    'lícia', 'alícia', 'felícia', 'luciana', 'larissa',
    'olá', 'bom dia', 'boa noite', 'obrigado', 'por favor',
    'ligar', 'desligar', 'acender', 'apagar', 'aumentar',
    'diminuir', 'temperatura', 'música', 'que horas são',
    'televisão', 'computador', 'celular', 'internet', 'cozinha',
]

base = f'./my_custom_model/{model_name}'
dirs = {
    'positive_train': f'{base}/positive_train',
    'positive_test':  f'{base}/positive_test',
    'negative_train': f'{base}/negative_train',
    'negative_test':  f'{base}/negative_test',
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

def gen_one_clip(args):
    """Gera um único clip (para uso em paralelo)."""
    word, voice, out_path = args
    try:
        subprocess.run(
            ['piper', '--model', voice, '--output_file', out_path,
             '--length-scale', str(random.choice(length_scales)),
             '--noise-scale', str(random.choice(noise_scales)),
             '--noise-w', str(random.choice(noise_ws))],
            input=word, capture_output=True, text=True, timeout=30
        )
        return os.path.exists(out_path)
    except Exception:
        return False

def gen_clips_parallel(text, out_dir, n, label, workers=4):
    """Gera clips em paralelo com ThreadPoolExecutor."""
    existing = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    if existing >= int(n * 0.95):
        print(f'  [SKIP] {label}: {existing} clips já existem')
        return
    needed = n - existing
    texts = [text] if isinstance(text, str) else text

    # Preparar lista de tarefas
    tasks = []
    for i in range(needed):
        word = random.choice(texts) if isinstance(texts, list) else texts
        voice = random.choice(ptbr_voices)
        out = os.path.join(out_dir, f'{uuid.uuid4().hex}.wav')
        tasks.append((word, voice, out))

    count = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(gen_one_clip, t): t for t in tasks}
        for future in as_completed(futures):
            if future.result():
                count += 1
            if count % 200 == 0 and count > 0:
                elapsed = time.time() - t0
                rate = count / elapsed
                remaining = (needed - count) / rate if rate > 0 else 0
                print(f'  {label}: {count}/{needed} ({rate:.1f} clips/s, ~{remaining/60:.0f} min)')

    total = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    elapsed = time.time() - t0
    print(f'  [OK] {label}: {total} clips ({elapsed/60:.1f} min)')

# Gerar com 4 workers paralelos (~4x mais rápido)
N_WORKERS = 4
print(f'\nUsando {N_WORKERS} workers paralelos')
print()
gen_clips_parallel(target_word,    dirs['positive_train'], n_positive, 'Positivos treino', N_WORKERS)
gen_clips_parallel(target_word,    dirs['positive_test'],  n_val,      'Positivos val', N_WORKERS)
gen_clips_parallel(negative_words, dirs['negative_train'], n_positive, 'Negativos treino', N_WORKERS)
gen_clips_parallel(negative_words, dirs['negative_test'],  n_val,      'Negativos val', N_WORKERS)

print()
print('Resumo:')
for label, d in dirs.items():
    n = len([f for f in os.listdir(d) if f.endswith('.wav')])
    print(f'  {label}: {n} clips')
print()
print('[OK] ETAPA 4 CONCLUÍDA!')


  ETAPA 4: Gerando clips com Piper pt_BR

Usando 4 workers paralelos

  Positivos treino: 200/1500 (0.5 clips/s, ~47 min)
  Positivos treino: 400/1500 (0.5 clips/s, ~40 min)
  Positivos treino: 600/1500 (0.5 clips/s, ~33 min)
  Positivos treino: 800/1500 (0.5 clips/s, ~25 min)
  Positivos treino: 1000/1500 (0.5 clips/s, ~18 min)
  Positivos treino: 1200/1500 (0.5 clips/s, ~11 min)
  Positivos treino: 1400/1500 (0.5 clips/s, ~4 min)
  [OK] Positivos treino: 1500 clips (54.1 min)
  Positivos val: 200/500 (0.5 clips/s, ~11 min)
  Positivos val: 400/500 (0.4 clips/s, ~4 min)
  [OK] Positivos val: 500 clips (18.6 min)
  Negativos treino: 200/1500 (0.4 clips/s, ~48 min)
  Negativos treino: 400/1500 (0.4 clips/s, ~41 min)
  Negativos treino: 600/1500 (0.5 clips/s, ~33 min)
  Negativos treino: 800/1500 (0.5 clips/s, ~26 min)
  Negativos treino: 1000/1500 (0.5 clips/s, ~18 min)
  Negativos treino: 1200/1500 (0.5 clips/s, ~11 min)
  Negativos treino: 1400/1500 (0.5 clips/s, ~4 min)
  [OK] Negati

## Etapa 5: Augmentação + Extração de Features

Usa o `train.py --augment_clips` do openWakeWord.

O que faz internamente (linhas 767-825 do train.py):
1. Lê clips de `positive_train/`, `positive_test/`, `negative_train/`, `negative_test/`
2. Aplica augmentação: convolução com RIR + mistura com background noise
3. Extrai features usando o embedding model
4. Salva `.npy` files

**Tempo estimado: ~15-30 minutos**

In [5]:
import os, sys, yaml

print('=' * 60)
print('  ETAPA 5: Augmentação e extração de features')
print('=' * 60)

model_name = 'leticia'

# RIR: preferir mit_rirs (271 arquivos), fallback para impulses/ (8 arquivos)
if os.path.exists('mit_rirs') and len([f for f in os.listdir('mit_rirs') if f.endswith('.wav')]) > 0:
    rir_path = os.path.abspath('mit_rirs')
else:
    rir_path = os.path.abspath('piper-sample-generator/impulses')
print(f'RIR: {rir_path} ({len(os.listdir(rir_path))} arquivos)')

# ============================================================
# Config YAML
# Campos obrigatórios do train.py (verificados lendo train.py linhas 600-920):
#   model_name, target_phrase, n_samples, n_samples_val,
#   tts_batch_size, augmentation_batch_size, piper_sample_generator_path,
#   output_dir, rir_paths, background_paths, background_paths_duplication_rate,
#   augmentation_rounds, false_positive_validation_data_path,
#   feature_data_files, batch_n_per_class, model_type, layer_size,
#   steps, max_negative_weight, target_false_positives_per_hour,
#   custom_negative_phrases
# ============================================================
config = {
    'model_name': model_name,
    'target_phrase': ['letícia'],
    'custom_negative_phrases': [
        'patrícia', 'notícia', 'delícia', 'justiça',
        'milícia', 'malícia', 'polícia', 'alícia', 'felícia'
    ],
    'n_samples': 1500,
    'n_samples_val': 500,
    'tts_batch_size': 50,
    'augmentation_batch_size': 16,
    'piper_sample_generator_path': os.path.abspath('./piper-sample-generator'),
    'output_dir': os.path.abspath('./my_custom_model'),
    'rir_paths': [rir_path],
    'background_paths': [
        os.path.abspath('./audioset_16k'),
        os.path.abspath('./fma')
    ],
    'background_paths_duplication_rate': [1, 1],
    'augmentation_rounds': 1,
    'false_positive_validation_data_path': os.path.abspath('./validation_set_features.npy'),
    'feature_data_files': {
        'ACAV100M_sample': os.path.abspath('./openwakeword_features_ACAV100M_2000_hrs_16bit.npy')
    },
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50
    },
    'model_type': 'dnn',
    'layer_size': 32,
    'steps': 50000,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.2,
}

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Config salva: my_model.yaml')
print()

# Executar augmentação
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# Verificar saída
feature_dir = f'my_custom_model/{model_name}'
expected_files = ['positive_features_train.npy', 'negative_features_train.npy',
                  'positive_features_test.npy', 'negative_features_test.npy']
print()
print('Features gerados:')
all_ok = True
for f in expected_files:
    fp = os.path.join(feature_dir, f)
    if os.path.exists(fp):
        shape = np.load(fp, mmap_mode='r').shape
        print(f'  ✅ {f}: {shape}')
    else:
        print(f'  ❌ {f}: NÃO ENCONTRADO')
        all_ok = False

if all_ok:
    print('\n[OK] ETAPA 5 CONCLUÍDA!')
else:
    print('\n[ERRO] Alguns features não foram gerados. Verifique os logs acima.')

  ETAPA 5: Augmentação e extração de features
RIR: /content/mit_rirs (270 arquivos)
Config salva: my_model.yaml

Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 4, in <module>
    import torchmetrics
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/__init__.py", line 14, in <module>
    from torchmetrics import functional  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/__init__.py", line 14, in <module>
    from torchmetrics.functional.audio._deprecated import _permutation_invariant_training as permutation_invariant_training
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/audio/__init__.py", line 14, in <module>
    from torchmetrics.functional.audio.pit import permutation_invariant_training, pit_permutate
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/audio/pit.py", line 22, in <module>
    from torchme

## Etapa 6: Treinar o Modelo

O `train.py --train_model` executa (linhas 827-918):
1. Cria DataLoaders com features (ACAV100M + clips gerados)
2. Treina DNN com 3 sequências (50K + 5K + 5K steps)
3. Merge top checkpoints via weight averaging
4. Exporta `.onnx` e `.tflite`

**Tempo estimado: ~30-60 min com GPU T4**

In [6]:
import sys

print('=' * 60)
print('  ETAPA 6: Treinando modelo')
print('=' * 60)

# --convert_to_tflite: usa onnx_tf internamente (train.py linha 916-918)
# Se falhar, a Etapa 7 tem fallback com onnx2tf
!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config my_model.yaml \
    --train_model \
    --convert_to_tflite

print()
print('[OK] ETAPA 6 CONCLUÍDA!')

  ETAPA 6: Treinando modelo
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 4, in <module>
    import torchmetrics
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/__init__.py", line 14, in <module>
    from torchmetrics import functional  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/__init__.py", line 14, in <module>
    from torchmetrics.functional.audio._deprecated import _permutation_invariant_training as permutation_invariant_training
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/audio/__init__.py", line 14, in <module>
    from torchmetrics.functional.audio.pit import permutation_invariant_training, pit_permutate
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/audio/pit.py", line 22, in <module>
    from torchmetrics.utilities import rank_zero_warn
  File "/usr/local/lib/python3.12/dist-packages

## Etapa 7: Verificar + Baixar o Modelo Final

In [7]:
import os, glob, shutil
from google.colab import files

print('=' * 60)
print('  ETAPA 7: Modelo final')
print('=' * 60)

model_name  = 'leticia'
onnx_path   = f'my_custom_model/{model_name}.onnx'
tflite_path = f'my_custom_model/{model_name}.tflite'

# Fallback: se train.py não gerou TFLite (onnx_tf pode falhar no Colab)
if os.path.exists(onnx_path) and not os.path.exists(tflite_path):
    print('TFLite não gerado pelo train.py. Tentando onnx2tf...')
    !onnx2tf -i {onnx_path} -o my_custom_model/tf_model -oiqt 2>&1
    tflite_files = glob.glob('my_custom_model/tf_model/**/*.tflite', recursive=True)
    if tflite_files:
        shutil.copy2(tflite_files[0], tflite_path)
        print(f'[OK] TFLite gerado via onnx2tf')
    else:
        print('[ERRO] Conversão TFLite falhou!')

# Relatório
print()
for path, label in [(onnx_path, 'ONNX'), (tflite_path, 'TFLite')]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f'  ✅ {label}: {path} ({size:.1f} KB)')
    else:
        print(f'  ❌ {label}: não encontrado')

# Download
print()
for path in [tflite_path, onnx_path]:
    if os.path.exists(path):
        print(f'Baixando {os.path.basename(path)}...')
        files.download(path)

print()
print('=' * 60)
print('  🎉 CONCLUÍDO!')
print()
print('  Deploy no Home Assistant Yellow:')
print('  1. Copie leticia.tflite para /share/openwakeword/')
print('  2. Reinicie o add-on openWakeWord')
print('  3. Configure: Assistants > Assist Pipeline > Wake Word = leticia')
print('  4. Teste: "Letícia, que horas são?"')
print('=' * 60)

  ETAPA 7: Modelo final

  ❌ ONNX: não encontrado
  ❌ TFLite: não encontrado


  🎉 CONCLUÍDO!

  Deploy no Home Assistant Yellow:
  1. Copie leticia.tflite para /share/openwakeword/
  2. Reinicie o add-on openWakeWord
  3. Configure: Assistants > Assist Pipeline > Wake Word = leticia
  4. Teste: "Letícia, que horas são?"
